# 1 · Text as a graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vitormeriat/text4gcn/blob/main/docs/tutorials/01_text_as_a_graph.ipynb)

A bag-of-words classifier looks at each document on its own. **TextGCN**
(Yao et al., 2019) takes a different view. The whole corpus becomes **one graph**,
whose nodes are the documents *and* the words, and a Graph Convolutional
Network learns by passing information along its edges.

Before training anything on a real dataset, we build that graph for **twelve
sentences** and open it up. By the end you will know what nodes and edges
Text4GCN creates, and why an unlabelled document can be classified just from
its position in the graph.

Runs on CPU in a few seconds.

| # | Tutorial | Question it answers |
|---|---|---|
| 1 | **Text as a graph** | What does the graph look like, and why does it help? |
| 2 | Your first classifier | How do I go from raw text to test accuracy? |
| 3 | Choosing a graph builder | How should words be connected? |
| 4 | Refining the graph | Can a smaller graph do as well? |
| 5 | Bring your own corpus | How do I use my own (non-English) data? |

In [ ]:
%pip install -q text4gcn
# Release candidate from TestPyPI (dependencies still come from PyPI):
# %pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "text4gcn==2.0.0rc2"

In [ ]:
import pickle
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from text4gcn import TextPipeline
from text4gcn.builder import Frequency
from text4gcn.datasets import save_corpus

## A tiny corpus

Two topics, *space* and *cooking*, with five labelled training sentences each.
The two **test** sentences have labels too, but the model will never see them:
we only use them to check the predictions at the end.

`save_corpus` writes the two files Text4GCN reads: `toy.txt`, with one document
per line, and `toy.meta`, with `<id>\t<train|test>\t<label>`.

In [ ]:
corpus = [
    ("The rocket launched the satellite into orbit around the planet", "space", "train"),
    ("Astronauts on the space station watched the planet from orbit", "space", "train"),
    ("The telescope observed a distant galaxy and its bright stars", "space", "train"),
    ("A new rocket engine will carry astronauts to the moon", "space", "train"),
    ("The moon and the stars shine above the space station", "space", "train"),
    ("Bake the bread in a hot oven until the crust is golden", "cooking", "train"),
    ("Chop the garlic and onion before adding them to the pan", "cooking", "train"),
    ("The chef seasoned the soup with garlic salt and fresh herbs", "cooking", "train"),
    ("Knead the dough then let the bread rise before you bake it", "cooking", "train"),
    ("Fry the onion in olive oil and add fresh herbs to the pan", "cooking", "train"),
    ("The satellite telescope photographed stars in a distant galaxy", "space", "test"),
    ("The chef baked fresh bread and served it with soup", "cooking", "test"),
]

NAME, DATA_DIR = "toy", Path("data")
shutil.rmtree(DATA_DIR, ignore_errors=True)

texts, labels, splits = zip(*corpus)
save_corpus(DATA_DIR, NAME, texts=texts, labels=labels, splits=splits)
print((DATA_DIR / f"{NAME}.meta").read_text(encoding="utf-8"))

## Step 1: preprocess

`TextPipeline` lower-cases and tokenizes the text and removes stop words. It
also drops words seen fewer than `rare_count` times; with twelve sentences we
keep every word, so `rare_count=1`. It then shuffles the documents and saves the
vocabulary.

In [ ]:
TextPipeline(dataset_name=NAME, dataset_path=DATA_DIR, rare_count=1).execute()

In [ ]:
shuffled = DATA_DIR / f"{NAME}.shuffled"
vocab = (shuffled / f"{NAME}.vocab").read_text(encoding="utf-8").split()
meta = pd.read_csv(shuffled / f"{NAME}.meta", sep="\t", names=["id", "split", "label"])
docs = (shuffled / f"{NAME}.txt").read_text(encoding="utf-8").splitlines()

print(f"{len(vocab)} words:", " ".join(vocab))
meta.assign(cleaned_text=docs)

## Step 2: build the graph

`Frequency.tfidf()` builds the original TextGCN graph and saves its adjacency
matrix as a `scipy.sparse` file. It returns `GraphBuildStats`, whose `path`
field points to that file.

In [ ]:
stats = Frequency(dataset_name=NAME, dataset_path=DATA_DIR).tfidf()
with open(stats.path, "rb") as f:
    adj = pickle.load(f)
stats

### Who is who

The adjacency matrix is square: **one row per node**. The nodes are in a fixed
order:

```
[ training documents | vocabulary words | test documents ]
```

In [ ]:
train, test = meta[meta.split == "train"], meta[meta.split == "test"]
n_train, n_words = len(train), len(vocab)
assert adj.shape[0] == n_train + n_words + len(test)

nodes = pd.DataFrame({
    "kind": ["train doc"] * n_train + ["word"] * n_words + ["test doc"] * len(test),
    "name": [f"{lb} doc #{i}" for i, lb in zip(train.id, train.label)]
            + vocab
            + [f"? doc #{i}" for i in test.id],
})
nodes.groupby("kind", sort=False).size().rename("nodes")

### Two kinds of edges

* **word–word**: two words that co-occur in the same sliding window more often
  than chance. The weight is their PMI (pointwise mutual information).
* **document–word**: a word that occurs in a document. The weight is its TF-IDF.

There are no document–document edges. Documents only meet **through shared
words**.

In [ ]:
coo = adj.maximum(adj.T).tocoo()   # symmetric: every edge in both directions
edges = pd.DataFrame({"src": nodes.name.values[coo.row], "dst": nodes.name.values[coo.col],
                      "src_kind": nodes.kind.values[coo.row], "dst_kind": nodes.kind.values[coo.col],
                      "weight": coo.data})
edges = edges[coo.row < coo.col]   # each undirected edge once

word_word = edges[(edges.src_kind == "word") & (edges.dst_kind == "word")]
print(f"{len(word_word)} word-word edges; the strongest (PMI):")
# With twelve sentences many pairs tie: they co-occur only in their single sentence.
word_word.nlargest(8, "weight")[["src", "dst", "weight"]]

In [ ]:
doc_word = edges[(edges.src_kind != "word") ^ (edges.dst_kind != "word")]
print(f"{len(doc_word)} document-word edges; the words of the first test document (TF-IDF):")
first_test = nodes.name.iloc[n_train + n_words]
doc_word[(doc_word.src == first_test) | (doc_word.dst == first_test)] \
    .sort_values("weight", ascending=False)[["src", "dst", "weight"]]

## Seeing the structure

We draw the graph with a small force-directed layout: edges pull their nodes
together, and all nodes push each other apart. Nothing here has been trained,
and labels are used only for the colours.

In [ ]:
def force_layout(linked: np.ndarray, steps: int = 300, seed: int = 0) -> np.ndarray:
    """Fruchterman-Reingold: linked nodes attract, every pair of nodes repels."""
    xy = np.random.default_rng(seed).normal(size=(len(linked), 2))
    k = 1 / np.sqrt(len(linked))
    for step in np.linspace(0.1, 0.002, steps):
        delta = xy[:, None] - xy[None, :]
        dist = np.linalg.norm(delta, axis=-1) + 1e-9
        force = ((k**2 / dist**2 - linked * dist / k)[..., None] * delta).sum(axis=1)
        length = np.linalg.norm(force, axis=1, keepdims=True) + 1e-9
        xy += force / length * np.minimum(length, step)
    return xy

dense = adj.toarray()
dense = np.maximum(dense, dense.T)                              # undirected
xy = force_layout(dense > 0)

colors = {"space": "tab:blue", "cooking": "tab:orange"}
fig, ax = plt.subplots(figsize=(10, 7))
for i, j in zip(*np.nonzero(np.triu(dense))):
    ax.plot(*xy[[i, j]].T, color="lightgray", lw=0.5, zorder=0)
for k, (kind, name) in enumerate(zip(nodes.kind, nodes.name)):
    if kind == "word":
        ax.annotate(name, xy[k], fontsize=8, color="dimgray", ha="center")
    else:
        label = meta.label.iloc[k if kind == "train doc" else k - n_words]
        ax.scatter(*xy[k], s=180, color=colors[label], edgecolor="black",
                   marker="o" if kind == "train doc" else "*", zorder=2)
ax.set_title("Word-document graph (circles: training documents, stars: test documents)")
ax.axis("off");

The two topics separate **before any training**. In this toy corpus they
share no words at all, so they form two islands; in real corpora common words
link the topics, and the edge weights carry the signal. Each test document (★)
sits among the training documents of its topic, connected through shared words
such as *bread*, *chef* or *telescope*.

## Why this helps: labels travel along edges

A 2-layer GCN computes `Â · ReLU(Â · X · W₁) · W₂`, where `Â` is the
normalised adjacency we just drew. Every layer mixes each node with its
neighbours, so after two layers a document has heard from its words *and* from
the other documents that share those words.

We can see the effect without any learning. Spread the one-hot labels of the
training documents over two hops (`Â² · Y`) and read the scores of the test
documents:

In [ ]:
a_hat = dense + np.eye(len(dense))                    # self-loops
d = a_hat.sum(axis=1)
a_hat = a_hat / np.sqrt(np.outer(d, d))              # D^-1/2 (A + I) D^-1/2

classes = sorted(colors)
y = np.zeros((len(dense), len(classes)))
for k, label in enumerate(train.label):
    y[k, classes.index(label)] = 1.0          # only training documents carry a label

scores = a_hat @ a_hat @ y                    # two hops of propagation
test_rows = range(n_train + n_words, len(dense))
pd.DataFrame(scores[test_rows], columns=classes,
             index=[f"{lb} (true)" for lb in test.label]).round(3)

Each test document gets the higher score for its true topic. The
labels reached it through the graph.

A GCN learns *how much* each hop and each feature should count, instead of
using fixed weights. Because test documents are nodes of the same graph, the
model is **transductive**: classifying a new document means adding it to the
graph and training again.

**Next:** [2 · Your first classifier](https://vitormeriat.github.io/text4gcn/tutorials/02_first_classifier/) runs the same three steps on a real
benchmark and trains the GCN.